In [1]:
import os
import sys
sys.path.append("kaggle/input/polymer_pipeline")

In [2]:
from data_preparation import get_data_paths, load_and_split_data
import model

In [3]:
os.environ['NEURIPS_DATA_PATH']     = 'kaggle/input/neurips-open-polymer-prediction-2025'
os.environ['EXTRA_DATA_BASE']       = 'kaggle/input/smiles-extra-data'
os.environ['TC_DATA_BASE']          = 'kaggle/input/tc-smiles'

In [4]:
paths = get_data_paths()
for k, v in paths.items():
    print(f"{k}: {v}")

train_csv: kaggle/input/neurips-open-polymer-prediction-2025/train.csv
test_csv: kaggle/input/neurips-open-polymer-prediction-2025/test.csv
sample_submission: kaggle/input/neurips-open-polymer-prediction-2025/sample_submission.csv
tc_data: kaggle/input/tc-smiles/Tc_SMILES.csv
tg_jcim_data: kaggle/input/smiles-extra-data/JCIM_sup_bigsmiles.csv
tg_excel_data: kaggle/input/smiles-extra-data/data_tg3.xlsx
density_data: kaggle/input/smiles-extra-data/data_dnst1.xlsx
supplement_dir: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement
ffv_data: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset4.csv
dataset1: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset1.csv
dataset2: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset2.csv
dataset3: kaggle/input/neurips-open-polymer-prediction-2025/train_supplement/dataset3.csv


In [5]:
train_df, val_df, test_df = load_and_split_data(paths)
print("Loaded:", len(train_df), len(val_df), len(test_df))

原始训练: 7973 条
  → 正在增强 Tc 数据，共 874 条
cross_smiles: 737 | 填充: 0
新增样本: 129 条
  → 正在增强 Tg 数据，共 662 条
cross_smiles: 526 | 填充: 15
新增样本: 136 条
  → 正在增强 Tg 数据，共 501 条
cross_smiles: 0 | 填充: 0
新增样本: 499 条
  → 正在增强 Density 数据，共 787 条


[02:38:37] SMILES Parse Error: syntax error while parsing: *O[Si](*)([R])[R]
[02:38:37] SMILES Parse Error: Failed parsing SMILES '*O[Si](*)([R])[R]' for input: '*O[Si](*)([R])[R]'
[02:38:37] SMILES Parse Error: syntax error while parsing: *NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4
[02:38:37] SMILES Parse Error: Failed parsing SMILES '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4' for input: '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4'
[02:38:37] SMILES Parse Error: syntax error while parsing: O=C=N[R1]N=C=O.O[R2]O.O[R3]O
[02:38:37] SMILES Parse Error: Failed parsing SMILES 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O' for input: 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O'
[02:38:37] SMILES Parse Error: syntax error while parsing: *CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O
[02:38:37] SMILES Parse Error: Failed parsing SMILES '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O' for input: '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O'
[02:38:37] SMILES Parse 

cross_smiles: 254 | 填充: 110
新增样本: 525 条
  → 正在增强 FFV 数据，共 862 条
cross_smiles: 43 | 填充: 43
新增样本: 819 条
Loaded: 8064 1008 1009


In [6]:
from train_stage1 import (setup_stage1_data, create_stage1_model, 
                        optimize_stage1, train_final_stage1_model,
                        optimize_stage1_debug,
                        quick_debug_test_fixed,
                        optimize_stage1_fixed)

/usr/local/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
print(train_df.head())

             id                                             SMILES  Tg  \
0  2.026603e+09    *Oc1ccc(CC2(Cc3ccc(*)cc3)c3ccccc3-c3ccccc32)cc1 NaN   
1  1.189403e+09         *CC(O)COc1ccc(C(C)CC(C)(C)c2ccc(O*)cc2)cc1 NaN   
2  1.686537e+09  *C(=O)c1ccc2c(c1)C(=O)N(c1c(C)cc(C(c3cc(C)c(N4... NaN   
3  2.632934e+08  *Oc1ccc2ccc(Oc3ccc(C(=Nc4ccc(N=C(c5ccccc5)c5cc... NaN   
4  1.280165e+09                    *Nc1ccc(-c2ccc(N*)c(OC)c2)cc1OC NaN   

        FFV  Tc  Density  Rg  
0  0.386736 NaN      NaN NaN  
1  0.354235 NaN      NaN NaN  
2  0.430396 NaN      NaN NaN  
3  0.381740 NaN      NaN NaN  
4  0.334115 NaN      NaN NaN  


In [9]:
# 运行Optuna优化
study = optimize_stage1(
    train_df,
    study_name="new_stage1_graph_ssl",
    n_trials=50,  # 可以根据需要调整
    patience=15
)

# 查看最佳参数
print("Best trial:")
print(" Value: ", study.best_trial.value)
print(" Params: ")
for key, value in study.best_trial.params.items():
    print(f"   {key}: {value}")

📦 构建 PolymerDataset，样本数=8064


   成功转换为图数据: 8064 条


[I 2025-08-02 02:25:12,125] Using an existing study with name 'new_stage1_graph_ssl' instead of creating a new one.


------------------------------------------
Epoch   Loss        Improvement Status    
------------------------------------------
1       nan         nan         🚫 No improve (1/15)
2       nan         nan         🚫 No improve (2/15)
3       nan         nan         🚫 No improve (3/15)
4       nan         nan         🚫 No improve (4/15)
5       nan         nan         🚫 No improve (5/15)
6       nan         nan         🚫 No improve (6/15)
7       nan         nan         🚫 No improve (7/15)
8       nan         nan         🚫 No improve (8/15)
9       nan         nan         🚫 No improve (9/15)
10      nan         nan         🚫 No improve (10/15)
11      nan         nan         🚫 No improve (11/15)
12      nan         nan         🚫 No improve (12/15)
13      nan         nan         🚫 No improve (13/15)
14      nan         nan         🚫 No improve (14/15)


[W 2025-08-02 02:26:20,401] Trial 1 failed with parameters: {'lr': 0.0040337851265135095, 'hidden_dim': 256, 'num_edge_layers': 7} because of the following error: ValueError('inf is not in list').
Traceback (most recent call last):
  File "/usr/local/miniconda3/lib/python3.12/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/root/kaggle-NeruIPS/kaggle/input/polymer_pipeline/train_stage1.py", line 212, in objective
    best_wmae, _ = train_stage1_model(
                   ^^^^^^^^^^^^^^^^^^^
  File "/root/kaggle-NeruIPS/kaggle/input/polymer_pipeline/train_stage1.py", line 168, in train_stage1_model
    print(f"Best loss: {best_loss:.4e} at epoch {history.index(best_loss)+1}")
                                                 ^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: inf is not in list
[W 2025-08-02 02:26:20,404] Trial 1 failed with value None.


15      nan         nan         🚫 No improve (15/15)
Early stopping at epoch 15
------------------------------------------


ValueError: inf is not in list

In [1]:
param_files = [f for f in os.listdir("stage1_artifacts") if f.startswith("stage1_best_params")]
print(param_files)

NameError: name 'os' is not defined

In [ ]:
import torch
latest_file = sorted(param_files)[-1]
params = torch.load(os.path.join("stage1_artifacts", latest_file))

In [ ]:
model = train_final_stage1_model(
    train_df,
    params=params,  # 可自动从文件加载
    output_path="production_model.pth"
    n_epochs=100,
    patience=15
)

📦 构建 PolymerDataset，样本数=8064


   成功转换为图数据: 8064 条
------------------------------------------
Epoch   Loss        Improvement Status    
------------------------------------------
1       5.5470e+00  inf         ✅ Improved
2       8.1183e-01  4.74e+00    ✅ Improved
3       6.4835e-01  1.63e-01    ✅ Improved
4       5.5532e-01  9.30e-02    ✅ Improved
5       4.9421e-01  6.11e-02    ✅ Improved
6       4.6085e-01  3.34e-02    ✅ Improved
7       4.3643e-01  2.44e-02    ✅ Improved
8       4.1437e-01  2.21e-02    ✅ Improved
9       3.9489e-01  1.95e-02    ✅ Improved
10      3.9723e-01  -2.33e-03   🚫 No improve (1/20)
11      3.9584e-01  -9.46e-04   🚫 No improve (2/20)
12      3.7540e-01  1.95e-02    ✅ Improved
13      3.8269e-01  -7.29e-03   🚫 No improve (1/20)
14      3.8292e-01  -7.52e-03   🚫 No improve (2/20)
15      3.7911e-01  -3.71e-03   🚫 No improve (3/20)
16      3.6398e-01  1.14e-02    ✅ Improved
17      3.6128e-01  2.70e-03    ✅ Improved
18      3.6218e-01  -8.98e-04   🚫 No improve (1/20)
19      3.6685e-01  -5.

In [ ]:
from rdkit.Chem import rdchem

# Atom 类支持的方法和属性
print([m for m in dir(rdchem.Atom) if m.startswith("Get")])

# Bond 类支持的方法和属性
print([m for m in dir(rdchem.Bond) if m.startswith("Get")])

['GetAtomMapNum', 'GetAtomicNum', 'GetBonds', 'GetBoolProp', 'GetChiralTag', 'GetDegree', 'GetDoubleProp', 'GetExplicitBitVectProp', 'GetExplicitValence', 'GetFormalCharge', 'GetHybridization', 'GetIdx', 'GetImplicitValence', 'GetIntProp', 'GetIsAromatic', 'GetIsotope', 'GetMass', 'GetMonomerInfo', 'GetNeighbors', 'GetNoImplicit', 'GetNumExplicitHs', 'GetNumImplicitHs', 'GetNumRadicalElectrons', 'GetOwningMol', 'GetPDBResidueInfo', 'GetProp', 'GetPropNames', 'GetPropsAsDict', 'GetQueryType', 'GetSmarts', 'GetSymbol', 'GetTotalDegree', 'GetTotalNumHs', 'GetTotalValence', 'GetUnsignedProp']
['GetBeginAtom', 'GetBeginAtomIdx', 'GetBondDir', 'GetBondType', 'GetBondTypeAsDouble', 'GetBoolProp', 'GetDoubleProp', 'GetEndAtom', 'GetEndAtomIdx', 'GetIdx', 'GetIntProp', 'GetIsAromatic', 'GetIsConjugated', 'GetOtherAtom', 'GetOtherAtomIdx', 'GetOwningMol', 'GetProp', 'GetPropNames', 'GetPropsAsDict', 'GetSmarts', 'GetStereo', 'GetStereoAtoms', 'GetUnsignedProp', 'GetValenceContrib']
